In [0]:
%pip install databricks-feature-engineering xgboost --quiet

%restart_python

In [0]:
import xgboost as xgb

from databricks.feature_engineering import FeatureEngineeringClient, FeatureLookup
import mlflow
import pyspark.sql.functions as F
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from mlflow.tracking import MlflowClient
from sklearn.ensemble import RandomForestRegressor
from mlflow.models.signature import infer_signature
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder


In [0]:
mlflow.xgboost.autolog(log_models=False)


In [0]:
mlflow.set_registry_uri('databricks-uc')

In [0]:
mlflow.set_experiment('/Users/abuhuzaifakurla@gmail.com/exp_daily_oil_gas_demand')

In [0]:
catalog_name = "ct_oil_gas" 
gold_schema_name = "sc_gold"
eval_split_table_name = "demand_eval_split"
feature_table_name = "daily_demand_features"
demand_labels_table_name = "daily_demand_labels"

RANDOM_STATE=  42

In [0]:
fe = FeatureEngineeringClient()

In [0]:
labels_df = spark.table(f"{catalog_name}.{gold_schema_name}.{demand_labels_table_name}")
split_df = spark.table(f"{catalog_name}.{gold_schema_name}.{eval_split_table_name}")

In [0]:
labels_df.printSchema()

In [0]:
import pyspark.sql.functions as F 

In [0]:
labels_df = labels_df.select(
    F.col("total_demand").cast("double"),
    "p_key",
    "transaction_date",
    "split")

In [0]:
labels_df.printSchema()

In [0]:
labels_df.filter("split =='train'").show()

In [0]:
train_eval_df = labels_df.filter("split =='train'").select("p_key","transaction_date","total_demand")

In [0]:
train_eval_df.show()

In [0]:

feature_lookups = [
    FeatureLookup(
        table_name=f"{catalog_name}.{gold_schema_name}.{feature_table_name}",
        lookup_key="p_key",
        timestamp_lookup_key="transaction_date",
        
    )
]

In [0]:
training_set = fe.create_training_set(
    df=train_eval_df,
    feature_lookups=feature_lookups,
    label="total_demand",
    exclude_columns=["p_key"]
)




training_df = training_set.load_df()

training_df.show()

In [0]:
training_df.printSchema()

In [0]:
# training_df = training_df.select(
#     F.col("avg_unit_price").cast("double"),
#     F.col("total_inventory").cast("double"),
#     F.col("demand_lag_1").cast("double"),
#     F.col("demand_lag_7").cast("double"),
#     F.col("demand_ma_7").cast("double"),
#     "total_demand",
#     "product_name",
#     "destination_city",
#     "transaction_date"
#     )

In [0]:
training_df.printSchema()

In [0]:
training_df.columns

In [0]:
training_pdf = training_df.toPandas()

In [0]:
training_pdf.shape

In [0]:
training_pdf.dtypes

In [0]:
training_pdf.head()

In [0]:
print(training_pdf.shape)

In [0]:
training_pdf_fill_na = training_pdf.dropna()

In [0]:
X = training_pdf_fill_na.drop("total_demand",axis=1)
y = training_pdf_fill_na.total_demand

X.shape , y.shape

In [0]:
X.dtypes

In [0]:
y.dtypes

In [0]:
X.isna().sum()

In [0]:
y = y.loc[X.index]



sorted_idx = X["transaction_date"].sort_values().index


X_sorted = X.loc[sorted_idx]

y_sorted = y.loc[sorted_idx]

# 80% train- 20% test 
split_point = int(len(X_sorted) * 0.8)

X_train = X_sorted.iloc[:split_point].drop(columns=["transaction_date"])
X_test  = X_sorted.iloc[split_point:].drop(columns=["transaction_date"])

y_train = y_sorted.iloc[:split_point]
y_test  = y_sorted.iloc[split_point:]

In [0]:
# numeric_cols = [
#     "avg_unit_price",
#     'total_inventory',
#     'demand_lag_1',
#     'demand_lag_7',
#     'demand_ma_7'
# ]

# for c in numeric_cols:
#     X_train[c] = X[c].astype(float)
#     X_test[c] = X[c].astype(float)


In [0]:
print(X_train.dtypes)
print(X_test.dtypes)
print(y_train.dtypes)
print(y_test.dtypes)



In [0]:
print(f"Train: {X_train.shape}, Test: {X_test.shape}, Y_train: {y_train.shape}, Y_test: {y_test.shape}")

In [0]:
import numpy as np

In [0]:


def to_str(df):
    df = df.copy()
    df[cat_cols] = df[cat_cols].astype(str)
    return df


In [0]:


cat_cols = ["product_name", "destination_city"]

X_train_s, X_test_s = to_str(X_train), to_str(X_test)

pre = ColumnTransformer(
    [("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=np.nan), cat_cols)],
    remainder="passthrough",
    verbose_feature_names_out=False,
).set_output(transform="pandas")

model = Pipeline([
    ("pre", pre),
    ("xgb", xgb.XGBRegressor(
        n_estimators=500, max_depth=7, learning_rate=0.05,
        tree_method="hist", random_state=RANDOM_STATE,
    )),
])

with mlflow.start_run(run_name="daily_demand_xgb") as run:
    model.fit(X_train_s, y_train)                      

    mlflow.log_params(model["xgb"].get_params())       

    y_pred = model.predict(X_test_s)                   
    val_mae = mean_absolute_error(y_test, y_pred)
    val_rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    val_r2 = r2_score(y_test, y_pred)
    mlflow.log_metrics({"val_mae": val_mae, "val_rmse": val_rmse, "val_r2": val_r2})

    fe.log_model(
        model=model,
        artifact_path="daily_oil_gas_demand_model",
        flavor=mlflow.sklearn,
        training_set=training_set,
        registered_model_name=None,
        input_example=X_train_s.iloc[[0]],
        signature = infer_signature(X_train_s, y_pred)

    )
    print(f"MAE {val_mae:.2f}, RMSE {val_rmse:.2f}, R2 {val_r2:.4f}, run_id {run.info.run_id}")

In [0]:
print(f"MAE {val_mae}, r2_score {val_r2}, RMSE {val_rmse}")